# Кандидатогенерация для поиска услуг

Генерируем 500 кандидатов для реранкера.

## Подход

6 фильтров на объявление. Первый — жёсткий,
остальные пять — мягкие, дают каждый своё число и объединяются весами
через **Reciprocal Rank Fusion**.

1. **Категория.** Объявление рассматривается только если
   `item_category_id` совпадает с `search_category` запроса. В трейне
   совпадение в 99.99% случаев.

2. **`coverage(q, d)` параметров.** Доля токенов запроса
   (`search_infm_params_text`), найденных в параметрах объявления
   (`item_infm_params_text`) — точным совпадением или (если точного нет)
   семантически, через эмбеддинги `multilingual-e5-large`.

3. **`coverage(q, d)` текста.** Точное совпадение токенов запроса в
   заголовке+описании объявления.

4. **Плотная E5-похожесть текста.** Косинусная близость эмбеддингов
   запроса и заголовка+начала описания объявления.

5. **Локация.** `feature_analysis.ipynb` показал, что 83%
   выбранных объявлений — в той же локации (`search_location_id ==
   item_location_id`). При несовпадении id объявление получает не 0, а
   убывающую с расстоянием оценку до центроида координат нужной локации
   (`LocationMatchIndex`, `src/location.py`) — соседние, но не идентичные
   локации не обнуляются полностью.

6. **Микрокатегория.** `item_category_id` почти не делит корпус (99% —
   категория 114), а внутри неё до ~200 `item_microcat_id`
   (`feature_analysis.ipynb`). У запроса нет явного поля микрокатегории, но
   у нас уже есть плотные E5-эмбеддинги объявлений — центроид эмбеддингов
   объявлений каждой микрокатегории и его косинусная близость к эмбеддингу
   запроса дают сигнал (`MicrocatMatchIndex`, `src/microcat.py`).

Все токенизаторы (для coverage параметров, coverage текста и общего
словаря) используют **лемматизацию** (`pymorphy3`).

## 0. Импорты и конфигурация

In [ ]:
import sys
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, ".")  # запускать из корня репозитория

from src.candidate_generator import CandidateGenerator
from src.config import CandidateGeneratorConfig
from src.data_loading import (
    build_local_eval_corpus,
    build_relevance_labels,
    load_dataset,
    sample_validation_queries,
)
from src.embeddings import E5Embedder
from src.evaluation import recall_at_k, relevant_item_coverage
from src.pipeline import build_indices
from src.text_processing import tokenize, unique_tokens

pd.set_option("display.max_colwidth", 120)

DATA_DIR = "dataset"
MODEL_DIR = "models/multilingual-e5-large"
CACHE_DIR = "cache/embeddings"
PARAMS_TRUNCATE_CHARS = 200    # первые ~200 символов params обычно содержат ключевые поля (Вид/Тип услуги)
DESCRIPTION_TRUNCATE_CHARS = 200  # для плотного title+description эмбеддинга

config = CandidateGeneratorConfig(top_k=500)
config

## 1. Загрузка данных

In [24]:
data = load_dataset(DATA_DIR)
for name, df in data.items():
    print(f"{name}: {len(df):>8,} строк, колонки: {list(df.columns)}")

train:  497,673 строк, колонки: ['search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category', 'item_id', 'item_title_raw', 'item_description_raw', 'item_infm_params_text', 'item_category_id', 'item_microcat_id', 'item_location_id', 'item_latitude', 'item_longitude']
benchmark_queries:    2,452 строк, колонки: ['query_id', 'search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category']
benchmark_items:  189,212 строк, колонки: ['item_id', 'item_title_raw', 'item_description_raw', 'item_infm_params_text', 'item_category_id', 'item_microcat_id', 'item_location_id', 'item_latitude', 'item_longitude']


In [25]:
data["benchmark_queries"].head(3)

,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114


## 2. Валидационная выборка

Для локальной проверки строим размеченный набор из `train.parquet`: каждая строка там — это пара "запрос → объявление, которое выбрал пользователь", так что если сгруппировать строки по (тексту запроса, параметрам, категории) и собрать все выбранные `item_id`, получится набор релевантных объявлений на запрос.

In [26]:
train = data["train"]

labels = build_relevance_labels(train)
print(f"Уникальных запросов (по тексту+параметрам+категории): {len(labels):,}")
print(labels["relevant_item_ids"].map(len).describe())

Уникальных запросов (по тексту+параметрам+категории): 107,978
count    107978.000000
mean          4.282659
std          30.220326
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        3714.000000
Name: relevant_item_ids, dtype: float64


In [27]:
N_VAL_QUERIES = 800

val_queries = sample_validation_queries(labels, n=N_VAL_QUERIES, seed=config.random_seed, max_relevant=200)
print(f"Валидационных запросов: {len(val_queries)}")
val_queries[["query_id", "search_query", "search_infm_params_text", "search_category"]].sample(5, random_state=1)

Валидационных запросов: 800


,query_id,search_query,search_infm_params_text,search_category
8,train_059527,поднятие года авто яндекс,,114
510,train_088486,спа для двоих,"Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",114
175,train_015386,восстановление безопасности,"Вид услуги Автосервис, аренда Тип услуги Автосервис",114
735,train_002760,автоподбор грузовых,"Вид услуги Автосервис, аренда",114
242,train_091632,текущая уборка,,114


In [28]:
eval_corpus = build_local_eval_corpus(data["benchmark_items"], train, val_queries)
print(f"Размер локального корпуса для оценки: {len(eval_corpus):,} "
      f"(из них добавлено сверх benchmark_items: {len(eval_corpus) - len(data['benchmark_items']):,})")

relevance = dict(zip(val_queries["query_id"], val_queries["relevant_item_ids"]))
coverage_check = relevant_item_coverage(relevance, eval_corpus["item_id"])
print(f"Доля релевантных объявлений, реально присутствующих в корпусе: {coverage_check:.4%}")
assert coverage_check == 1.0, "в корпус должны попасть все релевантные объявления валидационных запросов"

Размер локального корпуса для оценки: 191,311 (из них добавлено сверх benchmark_items: 2,099)
Доля релевантных объявлений, реально присутствующих в корпусе: 100.0000%


## 3. Выбор порога семантического совпадения параметров

`SemanticCoverageIndex` (`src/coverage.py`) сравнивает эмбеддинг **токена запроса** с эмбеддингом **всего текста параметров объявления**. 
Порог `similarity_threshold` откалиброван здесь на реальных парах "запрос/выбранное объявление" против случайных пар.

In [29]:
embedder = E5Embedder(MODEL_DIR, cache_dir=CACHE_DIR, batch_size=128, show_progress=True)

calib_sample = train[train["search_infm_params_text"].fillna("") != ""].sample(n=1500, random_state=0)

calib_query_vocab = sorted(unique_tokens(calib_sample["search_infm_params_text"]))
calib_query_emb = embedder.encode(calib_query_vocab, prefix="query", cache_key="calibration_query_vocab")
token_to_row = {token: i for i, token in enumerate(calib_query_vocab)}

calib_item_texts = calib_sample["item_infm_params_text"].fillna("").str.slice(0, PARAMS_TRUNCATE_CHARS).tolist()
calib_item_emb = embedder.encode(calib_item_texts, prefix="passage", cache_key="calibration_item_params")

rng = np.random.default_rng(0)
shuffled_idx = rng.permutation(len(calib_sample))
negative_item_emb = calib_item_emb[shuffled_idx]

In [30]:
positive_similarities = []
negative_similarities = []

for row_idx, (_, row) in enumerate(calib_sample.iterrows()):
    query_tokens = tokenize(row["search_infm_params_text"])
    token_rows = [token_to_row[t] for t in query_tokens if t in token_to_row]
    if not token_rows:
        continue
    token_embeddings = calib_query_emb[token_rows]
    positive_similarities.extend((token_embeddings @ calib_item_emb[row_idx]).tolist())
    negative_similarities.extend((token_embeddings @ negative_item_emb[row_idx]).tolist())

positive_similarities = np.array(positive_similarities)
negative_similarities = np.array(negative_similarities)

print(f"{len(positive_similarities):,} пар (токен запроса, params истинного объявления)")
print(f"{len(negative_similarities):,} пар (токен запроса, params случайного объявления)")

calibration_table = pd.DataFrame(
    {
        "threshold": (thresholds := np.round(np.arange(0.74, 0.90, 0.02), 2)),
        "true_positive_rate": [(positive_similarities >= t).mean() for t in thresholds],
        "false_positive_rate": [(negative_similarities >= t).mean() for t in thresholds],
    }
)
calibration_table["separation"] = calibration_table["true_positive_rate"] - calibration_table["false_positive_rate"]
calibration_table

10,256 пар (токен запроса, params истинного объявления)
10,256 пар (токен запроса, params случайного объявления)


,threshold,true_positive_rate,false_positive_rate,separation
0,0.74,0.997562,0.990542,0.007020
1,0.76,0.979427,0.910881,0.068545
2,0.78,0.827418,0.616712,0.210706
3,0.80,0.589606,0.292512,0.297094
4,0.82,0.231377,0.051677,0.179700
5,0.84,0.069715,0.007313,0.062402
6,0.86,0.000975,0.000390,0.000585
7,0.88,0.000000,0.000000,0.000000
8,0.90,0.000000,0.000000,0.000000


## 4. Индексы для кандидатогенерации

Сборка всех пяти индексов вынесена в `src/pipeline.build_indices` — она же используется в `reranking.ipynb`, чтобы оба ноутбука не расходились в деталях сборки.

Кодирование всего корпуса (`benchmark_items`, 189 212 объявлений) — самая дорогая операция во всей тетради, но результат кешируется на диск по хешу входных текстов (`E5Embedder._cache_path`), так что при повторном запуске ячейка отрабатывает мгновенно. Отдельно кодируем только ту небольшую добавку объявлений, которой не хватает в `benchmark_items`, но которая нужна для локальной валидации — это позволяет не кодировать 189k объявлений дважды.

In [32]:
t0 = time.time()
benchmark_items = data["benchmark_items"]

benchmark_dense_texts = (
    benchmark_items["item_title_raw"].fillna("")
    + " "
    + benchmark_items["item_description_raw"].fillna("").str.slice(0, DESCRIPTION_TRUNCATE_CHARS)
).tolist()
benchmark_dense_emb = embedder.encode(
    benchmark_dense_texts, prefix="passage", cache_key="benchmark_item_title_description"
)
benchmark_params_emb = embedder.encode(
    benchmark_items["item_infm_params_text"].fillna("").str.slice(0, PARAMS_TRUNCATE_CHARS).tolist(),
    prefix="passage",
    cache_key="benchmark_item_params",
)
print(f"benchmark_items закодированы за {time.time()-t0:.1f}s: "
      f"title+description {benchmark_dense_emb.shape}, params {benchmark_params_emb.shape}")

benchmark_items закодированы за 1.5s: title+description (189212, 1024), params (189212, 1024)


In [33]:
n_benchmark_items = len(benchmark_items)
extra_items = eval_corpus.iloc[n_benchmark_items:]
print(f"Дополнительных объявлений (есть только в train, нет в benchmark_items): {len(extra_items):,}")

if len(extra_items):
    extra_dense_texts = (
        extra_items["item_title_raw"].fillna("")
        + " "
        + extra_items["item_description_raw"].fillna("").str.slice(0, DESCRIPTION_TRUNCATE_CHARS)
    ).tolist()
    extra_dense_emb = embedder.encode(extra_dense_texts, prefix="passage", cache_key="val_extra_item_title_description")
    extra_params_emb = embedder.encode(
        extra_items["item_infm_params_text"].fillna("").str.slice(0, PARAMS_TRUNCATE_CHARS).tolist(),
        prefix="passage",
        cache_key="val_extra_item_params",
    )
    eval_dense_emb = np.concatenate([benchmark_dense_emb, extra_dense_emb])
    eval_params_emb = np.concatenate([benchmark_params_emb, extra_params_emb])
else:
    eval_dense_emb = benchmark_dense_emb
    eval_params_emb = benchmark_params_emb

assert eval_dense_emb.shape[0] == len(eval_corpus)
assert eval_params_emb.shape[0] == len(eval_corpus)

Дополнительных объявлений (есть только в train, нет в benchmark_items): 2,099


In [34]:
eval_query_vocab = sorted(
    unique_tokens(val_queries["search_infm_params_text"]) | unique_tokens(data["benchmark_queries"]["search_infm_params_text"])
)
print(f"Словарь токенов параметров запросов (val + benchmark): {len(eval_query_vocab)}")

eval_params_index, eval_text_index, eval_dense_index, eval_location_index, eval_microcat_index = build_indices(
    eval_corpus, eval_params_emb, eval_dense_emb, eval_query_vocab, embedder, config,
    query_cache_key="eval_query_vocab_params",
)

eval_generator = CandidateGenerator(
    eval_corpus, eval_params_index, eval_text_index, eval_dense_index, eval_location_index, eval_microcat_index, config
)
print("Индексы для локальной валидации готовы.")

Словарь токенов параметров запросов (val + benchmark): 299
Индексы для локальной валидации готовы.


## 5. Кандидатогенерация для валидации и Recall@k

Генерируем до 500 кандидатов для каждого из `N_VAL_QUERIES` валидационных
запросов и считаем Recall@k при `k = 50, 100, 200, 250, 500`.

In [35]:
val_query_embeddings = embedder.encode(
    val_queries["search_query"].fillna("").tolist(), prefix="query", cache_key="val_query_text_emb"
)

t0 = time.time()
val_predictions = eval_generator.generate(val_queries, val_query_embeddings)
print(f"Кандидаты для {len(val_queries)} запросов сгенерированы за {time.time()-t0:.1f}s")

Generating candidates:   0%|          | 0/800 [00:00<?, ?it/s]

Кандидаты для 800 запросов сгенерированы за 53.1s


In [36]:
recall_summary, recall_per_query = recall_at_k(val_predictions, relevance, ks=(50, 100, 200, 250, 500))
recall_summary

,k,recall_at_k
0,50,0.511462
1,100,0.638434
2,200,0.754597
3,250,0.794390
4,500,0.891271


## 6. Вклад каждого из пяти мягких сигналов

Отключаем по одному сигналу за раз (вес в RRF — 0).

In [37]:
def run_ablation(weights: dict, query_subset: pd.DataFrame, query_embeddings: np.ndarray, label: str) -> dict:
    ablation_config = CandidateGeneratorConfig(top_k=config.top_k, signal_weights=weights)
    generator = CandidateGenerator(
        eval_corpus, eval_params_index, eval_text_index, eval_dense_index, eval_location_index, eval_microcat_index, ablation_config
    )
    predictions = generator.generate(query_subset, query_embeddings, show_progress=False)
    subset_relevance = {qid: relevance[qid] for qid in query_subset["query_id"]}
    summary, _ = recall_at_k(predictions, subset_relevance, ks=(50, 500))
    row = {"variant": label, "n_queries": len(query_subset)}
    row.update(dict(zip(summary["k"].map(lambda k: f"recall@{k}"), summary["recall_at_k"])))
    return row


FULL_WEIGHTS = {
    "params_coverage": 1.0, "text_coverage": 1.0, "text_similarity": 1.0,
    "location_match": 1.0, "microcat_match": 1.0,
}

signal_ablation_variants = {
    "все пять сигналов": FULL_WEIGHTS,
    "без coverage параметров": {**FULL_WEIGHTS, "params_coverage": 0.0},
    "без coverage текста": {**FULL_WEIGHTS, "text_coverage": 0.0},
    "без плотной похожести": {**FULL_WEIGHTS, "text_similarity": 0.0},
    "без локации": {**FULL_WEIGHTS, "location_match": 0.0},
    "без микрокатегории": {**FULL_WEIGHTS, "microcat_match": 0.0},
    "только coverage текста": {
        "params_coverage": 0.0, "text_coverage": 1.0, "text_similarity": 0.0,
        "location_match": 0.0, "microcat_match": 0.0,
    },
}

signal_ablation_results = pd.DataFrame(
    [
        run_ablation(weights, val_queries, val_query_embeddings, label)
        for label, weights in tqdm(signal_ablation_variants.items(), desc="Абляция сигналов")
    ]
)
signal_ablation_results

Абляция сигналов:   0%|          | 0/7 [00:00<?, ?it/s]

,variant,n_queries,recall@50,recall@500
0,все пять сигналов,800,0.511462,0.891271
1,без coverage параметров,800,0.509483,0.884808
2,без coverage текста,800,0.439596,0.833406
3,без плотной похожести,800,0.569868,0.850599
4,без локации,800,0.436266,0.813142
5,без микрокатегории,800,0.512283,0.879240
6,только coverage текста,800,0.279156,0.636429


## 7. Кандидатогенерация для теста

Тот же пайплайн — на `benchmark_queries.parquet` и полном `benchmark_items.parquet`.
Результат — до 500 кандидатов на запрос — сохраняется на диск и понадобится на следующем этапе (переранжирование до финальных 50).

In [38]:
benchmark_params_index, benchmark_text_index, benchmark_dense_index, benchmark_location_index, benchmark_microcat_index = build_indices(
    benchmark_items, benchmark_params_emb, benchmark_dense_emb, eval_query_vocab, embedder, config,
    query_cache_key="eval_query_vocab_params",
)
benchmark_generator = CandidateGenerator(
    benchmark_items, benchmark_params_index, benchmark_text_index, benchmark_dense_index,
    benchmark_location_index, benchmark_microcat_index, config
)

In [39]:
benchmark_query_embeddings = embedder.encode(
    data["benchmark_queries"]["search_query"].fillna("").tolist(),
    prefix="query",
    cache_key="benchmark_query_text_emb",
)

t0 = time.time()
benchmark_predictions = benchmark_generator.generate(data["benchmark_queries"], benchmark_query_embeddings)
print(f"Кандидаты для {len(data['benchmark_queries'])} запросов бенчмарка сгенерированы за {time.time()-t0:.1f}s")

Generating candidates:   0%|          | 0/2452 [00:00<?, ?it/s]

Кандидаты для 2452 запросов бенчмарка сгенерированы за 153.8s


In [40]:
import pathlib

output_path = pathlib.Path("cache/candidates_top500.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

candidates_df = pd.DataFrame(
    {"query_id": list(benchmark_predictions.keys()), "candidate_item_ids": list(benchmark_predictions.values())}
)
candidates_df["n_candidates"] = candidates_df["candidate_item_ids"].map(len)
candidates_df.to_parquet(output_path, index=False)

print(f"Сохранено: {output_path} ({len(candidates_df)} строк)")
candidates_df["n_candidates"].describe()

Сохранено: cache/candidates_top500.parquet (2452 строк)


count    2452.0
mean      500.0
std         0.0
min       500.0
25%       500.0
50%       500.0
75%       500.0
max       500.0
Name: n_candidates, dtype: float64